In [ ]:
!pip install -q diffusers transformers accelerate safetensors xformers gradio

In [ ]:
# Imports

import torch
from huggingface_hub import login
from IPython.display import display
from google.colab import userdata
from io import BytesIO
from diffusers import StableDiffusionXLPipeline
from diffusers import AutoencoderKL
from transformers import Blip2Processor, Blip2ForConditionalGeneration
from PIL import Image
from google.colab import files
import gradio as gr

In [ ]:
# Let's check the GPU - it should be a Tesla T4

gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
  if gpu_info.find('Tesla T4') >= 0:
    print("Success - Connected to a T4")
  else:
    print("NOT CONNECTED TO A T4")

In [ ]:
hf_token = userdata.get('HF_TOKEN')
if hf_token and hf_token.startswith("hf_"):
  print("HF key looks good so far")
else:
  print("HF key is not set - please click the key in the left sidebar")
login(hf_token, add_to_git_credential=True)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def load_caption_model():
    global loaded_model, current_model, model, processor

    processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    model = Blip2ForConditionalGeneration.from_pretrained(
        "Salesforce/blip2-opt-2.7b",
        torch_dtype=torch.float16
    ).to("cuda")

    loaded_model = model
    current_model = "caption"

In [ ]:
def load_image_model():
    global loaded_model, current_model, pipe

    vae = AutoencoderKL.from_pretrained(
        "madebyollin/sdxl-vae-fp16-fix",
        torch_dtype=torch.float16
    )
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
    ).to("cuda")
    pipe.enable_xformers_memory_efficient_attention()

    loaded_model = pipe
    current_model = "text-to-image"

In [ ]:
def unload_current_model():
    global loaded_model, current_model, model, processor, pipe

    if current_model == "caption":
        del model
        del processor
        del loaded_model
    elif current_model == "text-to-image":
        del pipe
        del loaded_model

    gc.collect()
    torch.cuda.empty_cache()
    current_model = None
    loaded_model = None

In [ ]:
def caption_image(image):
    prompt = "Describe this image in detail:"
    inputs = processor(images=image, text=prompt, return_tensors="pt").to("cuda", torch.float16)

    input_len = inputs["input_ids"].shape[1]

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        num_beams=3,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
    )

    caption = processor.decode(output[0][input_len:], skip_special_tokens=True)
    return caption

In [ ]:
def text_to_image(prompt):
    image = pipe(
        prompt=prompt,
        negative_prompt="photorealistic, blurry, low quality, distorted face, bad anatomy, extra fingers, deformed eyes, ugly, watermark, disfigured, floating limbs, cropped",
        height=768,
        width=768,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=torch.Generator("cuda").manual_seed(42)
    ).images[0]

    # Clean up after every generation
    gc.collect()
    torch.cuda.empty_cache()

    return image

In [ ]:
loaded_model = None
current_model = None

In [ ]:
import gradio as gr
import gc

def update_inputs(choice):
    if choice == "Text-To-Image":
        return (
            gr.update(visible=True),   # prompt
            gr.update(visible=False),  # image_input
            gr.update(visible=True),   # image_output
            gr.update(visible=False)   # caption_output
        )
    elif choice == "Image Captioning":
        return (
            gr.update(visible=False),  # prompt
            gr.update(visible=True),   # image_input
            gr.update(visible=False),  # image_output
            gr.update(visible=True)    # caption_output
        )
    else:
        return (
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=False)
        )


def run_model(model_choice, prompt, image):
    global current_model

    free_vram = torch.cuda.mem_get_info()[0] / 1024**3  # in GB
    if free_vram < 1.5:
        return None, f"⚠️ Low VRAM ({free_vram:.1f}GB free), clearing memory..."
        gc.collect()
        torch.cuda.empty_cache()

    if model_choice == "Text-To-Image":
        if current_model != "text-to-image":
            unload_current_model()
            load_image_model()
        result = text_to_image(prompt)
        return result, ""  # image output, no caption

    elif model_choice == "Image Captioning":
        if current_model != "caption":
            unload_current_model()
            load_caption_model()

        pil_image = Image.open(image).convert("RGB")
        caption = caption_image(pil_image)

        return None, caption  # no image output, caption text


with gr.Blocks() as demo:
    gr.Markdown("## DualMind 🧠🧠")

    with gr.Row():
        with gr.Column(scale=1):
            model_choice = gr.Dropdown(
                choices=["Text-To-Image", "Image Captioning"],
                label="Select a Model",
                value=None
            )
            prompt = gr.Textbox(label="Enter Prompt:", visible=False)
            image_input = gr.Image(type="filepath", label="Upload Image", visible=False)
            btn = gr.Button("Run", variant="primary")

        with gr.Column(scale=1):
            image_output = gr.Image(label="Generated Image", visible=False)
            caption_output = gr.Textbox(label="Caption", visible=False, lines=4)

    model_choice.change(
        fn=update_inputs,
        inputs=model_choice,
        outputs=[prompt, image_input, image_output, caption_output]
    )

    btn.click(
        fn=run_model,
        inputs=[model_choice, prompt, image_input],
        outputs=[image_output, caption_output]
    )

demo.launch(debug=True, share=True)